In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from students.marchenko.lesson3 import Exercise

In [ ]:
def normalize_data(X_train, X_val, X_test):

    # Вычисляем статистики только на тренировочных данных
    feature_means = X_train.mean(axis=0)
    feature_stds = X_train.std(axis=0)

    # Защита от константных признаков (std=0)
    feature_stds[feature_stds == 0] = 1.0

    # Нормализация всех выборок на основе train статистик
    X_train_norm = (X_train - feature_means) / feature_stds
    X_val_norm = (X_val - feature_means) / feature_stds
    X_test_norm = (X_test - feature_means) / feature_stds

    return X_train_norm, X_val_norm, X_test_norm

In [ ]:
print("=" * 60)
print("ЗАГРУЗКА ДАННЫХ")
print("=" * 60)

digits = load_digits()
X = digits.data.astype(np.float32)  #  64 признака (8×8 пикселей)
y = digits.target.astype(np.int64)  #  метки 0-9

print(f"Всего образцов: {X.shape[0]}")
print(f"Признаков: {X.shape[1]}")
print(f"Классов: {len(np.unique(y))}")
print(f"Диапазон пикселей: [{X.min()}, {X.max()}]")

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=100, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=100, stratify=y_temp)

print(
    f"\nРазмеры: Train {len(X_train)} ({len(X_train) / len(X) * 100:.0f}%), "
    f"Val {len(X_val)} ({len(X_val) / len(X) * 100:.0f}%), "
    f"Test {len(X_test)} ({len(X_test) / len(X) * 100:.0f}%)"
)

X_train_norm, X_val_norm, X_test_norm = normalize_data(X_train, X_val, X_test)

print("\nПосле нормализации:")
print(f"  Train mean: {X_train_norm.mean():.6f}, std: {X_train_norm.std():.6f}")
print(f"  Диапазон: [{X_train_norm.min():.2f}, {X_train_norm.max():.2f}]")

In [ ]:
MSELoss = Exercise.create_mse_loss
Model = Exercise.create_model
LinearLayer = Exercise.create_linear_layer
ReLULayer = Exercise.create_relu_layer
CrossEntropyLoss = Exercise.create_cross_entropy_loss
NLLloss = Exercise.create_nll_loss
LogSoftMax = Exercise.create_logsoftmax_layer


def evaluate_config(
    learning_rate, batch_size, X_train, X_val, y_train, y_val, n_ep=25, random_seed=100, hidden_size=32
):

    # Создаём модель с фиксированным seed
    rng = np.random.default_rng(random_seed)
    model = Model(
        LinearLayer(X_train.shape[1], hidden_size, rng=rng),
        ReLULayer(),
        LinearLayer(hidden_size, len(np.unique(y_train)), rng=rng),
    )
    loss = CrossEntropyLoss()

    # Обучаем
    Exercise.train_model(
        model=model,
        loss=loss,
        x=X_train,
        y=y_train,
        lr=learning_rate,
        n_epoch=n_ep,
        batch_size=batch_size if batch_size is not None else len(X_train),
        shuffle=False,
    )

    val_predictions = model.forward(X_val)
    val_predicted_classes = np.argmax(val_predictions, axis=1)
    accuracy = np.mean(val_predicted_classes == y_val)

    return accuracy, model


n_ep = 25
seeds = [100, 200, 300, 400]
lr_values = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1, 0.3, 0.5, 1.0]
batch_sizes = [1, 2, 4, 8, 16, 32, 64, None]
hidden_size = 32

results = []

print("\n" + "=" * 70)
print("ПОДБОР ГИПЕРПАРАМЕТРОВ")
print("=" * 70)

total_configs = len(lr_values) * len(batch_sizes)
config_num = 0

for lr in lr_values:
    for bs in batch_sizes:
        config_num += 1
        print(f"\rКонфигурация {config_num}/{total_configs}...", end="")

        acc_scores = []

        for seed in seeds:
            try:
                acc, _ = evaluate_config(lr, bs, X_train_norm, X_val_norm, y_train, y_val, n_ep, seed, hidden_size)
                acc_scores.append(acc)
            except Exception:
                acc_scores.append(0.0)

        results.append(
            {
                "lr": lr,
                "bs": bs,
                "acc_mean": np.mean(acc_scores),
                "acc_std": np.std(acc_scores),
            }
        )

print("\r" + " " * 50 + "\r", end="")

results.sort(key=lambda x: -x["acc_mean"])

print("\n" + "=" * 70)
print("ТОП-5 КОНФИГУРАЦИЙ ПО ACCURACY")
print("=" * 70)
print(f"{'lr':<8} {'batch':<8} {'hidden':<8} {'Accuracy':<16}")
print("-" * 70)

for r in results[:5]:
    bs = "None" if r["bs"] is None else str(r["bs"])
    print(f"{r['lr']:<8.3f} {bs:<8} {r['acc_mean']:.4f}±{r['acc_std']:.4f}")


best = results[0]

print("\n" + "=" * 70)
print("ЛУЧШАЯ КОНФИГУРАЦИЯ")
print("=" * 70)
print(f"Learning Rate:  {best['lr']}")
print(f"Batch Size:     {'None' if best['bs'] is None else best['bs']}")
print(f"Hidden Size:    {hidden_size}")
print(f"Val Accuracy:   {best['acc_mean']:.4f} ± {best['acc_std']:.4f}")

final_rng = np.random.default_rng(42)

final_model = Model(
    LinearLayer(X_train_norm.shape[1], hidden_size, rng=final_rng),
    ReLULayer(),
    LinearLayer(hidden_size, len(np.unique(y_train)), rng=final_rng),
)

Exercise.train_model(
    model=final_model,
    loss=CrossEntropyLoss(),
    x=X_train_norm,
    y=y_train,
    lr=best["lr"],
    n_epoch=n_ep,
    batch_size=best["bs"] if best["bs"] is not None else len(X_train_norm),
    shuffle=False,
)

test_predictions = final_model.forward(X_test_norm)
test_predicted_classes = np.argmax(test_predictions, axis=1)
test_acc = np.mean(test_predicted_classes == y_test)

print("\n" + "=" * 70)
print("ФИНАЛЬНЫЙ ТЕСТ")
print("=" * 70)
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Gap:            {best['acc_mean'] - test_acc:.4f}")